# Customer Support — Baseline Runs (Kaggle)

Runs directly from `customer_support_28k_fixed.csv` — no separate 500-row file needed.

**Before running:**
1. Notebook Settings → Accelerator → **GPU T4 x2**
2. Notebook Settings → Internet → **On**
3. Add-ons → Secrets → add `HF_TOKEN` (needed for LLaMA only)
4. Confirm `cs_dataset` and `finetuned-sbert-customer-support` are attached under Add-ons → Datasets

**Flow:**
```
A: env check  →  B: install  →  C: setup + sample  →  D: smoke test (10 rows)  →  E: full run (500 rows)
```

## A — Environment check

In [ ]:
import torch, os

if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU — Notebook Settings → Accelerator → GPU T4 x2')

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN: loaded')
except Exception:
    print('HF_TOKEN: not set — T5 + Gemma will still work, LLaMA needs it')

print('\nDataset check:')
for p in [
    '/kaggle/input/cs-dataset/customer_support_28k_fixed.csv',
    '/kaggle/input/finetuned-sbert-customer-support/',
]:
    print(f'  {"✅" if os.path.exists(p) else "❌ NOT FOUND"}  {p}')

## B — Install dependencies
torch, transformers, pandas already on Kaggle — only installing what's missing.

In [ ]:
%%capture
!pip install rouge-score bert-score nltk rank-bm25 faiss-cpu \
             sentence-transformers accelerate -q
!python -m spacy download en_core_web_sm -q
!python -m spacy download de_core_news_sm -q
print('Done.')

## C — Clone repo + create stratified sample from 28k file

In [ ]:
import os

WORKING  = '/kaggle/working'
REPO     = 'customer-support-response-nlp'
PIPELINE = f'{WORKING}/{REPO}/pipeline'

os.chdir(WORKING)
if not os.path.exists(f'{WORKING}/{REPO}'):
    !git clone https://github.com/ranjan56cse/{REPO}.git
else:
    !cd {WORKING}/{REPO} && git pull

os.chdir(PIPELINE)
print(f'Pipeline dir: {PIPELINE}')

In [ ]:
import pandas as pd

RAW_CSV  = '/kaggle/input/cs-dataset/customer_support_28k_fixed.csv'
SAMPLE_N = 500   # rows for the full baseline run
SEED     = 42

df_full = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Full dataset : {len(df_full):,} rows')
print(f'Languages    : {df_full["language"].value_counts().to_dict()}')
print(f'answer col   : {"answer" in df_full.columns}')

# Stratified sample — preserves EN/DE ratio from the full dataset
df_sample = (
    df_full
    .groupby('language', group_keys=False)
    .apply(lambda g: g.sample(
        min(len(g), round(SAMPLE_N * len(g) / len(df_full))),
        random_state=SEED
    ))
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
df_sample['idx'] = df_sample.index

SAMPLE_CSV = 'sample_500.csv'
df_sample.to_csv(SAMPLE_CSV, index=False)

print(f'\nSample saved : {SAMPLE_CSV}')
print(f'Rows         : {len(df_sample)}')
print(f'EN: {(df_sample["language"]=="en").sum()}  '
      f'DE: {(df_sample["language"]=="de").sum()}')

In [ ]:
# ── Run aspect pipeline to extract 6 aspects ─────────────────────────────
# Produces aspects_sample_500.csv
# Skip this cell if aspects_sample_500.csv already exists from a prior run.

import glob, os

existing = glob.glob('aspects_*.csv')
if existing:
    print(f'Aspects file already exists: {existing[0]} — skipping extraction')
    print('Delete it and re-run this cell if you want fresh aspects.')
else:
    print('Running aspect extraction on 500 rows (~10 min on T4)...')
    !python aspect_pipeline.py --input sample_500.csv
    produced = glob.glob('aspects_*.csv')
    print(f'\n{"✅ Done: " + str(produced) if produced else "❌ No aspects file produced — check errors above"}')

In [ ]:
# ── Verify aspects file ───────────────────────────────────────────────────
import pandas as pd, glob

ASPECTS_CSV = glob.glob('aspects_*.csv')[0]
df_asp = pd.read_csv(ASPECTS_CSV)

print(f'Aspects file : {ASPECTS_CSV}  ({len(df_asp)} rows)')
print(f'prob_sub     : {(df_asp["prob_sub"]!="none").sum()}/{len(df_asp)} filled')
print(f'prob_statement: {(df_asp["prob_statement"]!="none").sum()}/{len(df_asp)} filled')
print(f'cause        : {(df_asp["cause"]!="none").sum()}/{len(df_asp)} filled')
print(f'urgency_vibe : {df_asp["urgency_vibe"].value_counts().to_dict()}')

---
## D — RUN 1: Smoke test (10 rows, T5 only)
~5 minutes. No token needed. Confirms end-to-end before committing to the full run.

In [ ]:
import glob
ASPECTS_CSV = glob.glob('aspects_*.csv')[0]

!python response_generator.py \
    --aspects {ASPECTS_CSV} \
    --mode both \
    --n 10

In [ ]:
import pandas as pd, glob

mfiles = glob.glob('metrics_*.csv')
if mfiles:
    print('=== SMOKE TEST METRICS (10 rows) ===')
    print(pd.read_csv(mfiles[0]).to_string(index=False))
    print('\n✅ Smoke test passed — proceed to Section E')
else:
    print('❌ No metrics file — fix errors above before Section E')

bfiles = glob.glob('responses_*_baseline.csv')
if bfiles:
    row = pd.read_csv(bfiles[0]).iloc[0]
    print(f'\nSample response:')
    print(f'  Subject : {row["subject"]}')
    print(f'  Response: {str(row["baseline_response"])[:300]}')

---
## E — RUN 2: Full run (500 rows, all 4 models)
**Only after smoke test passes.**

| Model | Token | Est. time |
|---|---|---|
| T5 baseline | No | ~20 min |
| T5 RAG+Aspect | No | ~25 min |
| LLaMA-3.2-1B | Yes | ~25 min |
| Gemma-2-2B | No | ~25 min |
| **Total** | | **~95 min** |

In [ ]:
# Pre-flight check
import torch, os, glob

ASPECTS_CSV = glob.glob('aspects_*.csv')[0]
checks = {
    'GPU available'         : torch.cuda.is_available(),
    'Aspects CSV present'   : os.path.exists(ASPECTS_CSV),
    'HF_TOKEN set'          : bool(os.environ.get('HF_TOKEN')),
    'response_generator.py' : os.path.exists('response_generator.py'),
    'rag.py'                : os.path.exists('rag.py'),
}
for label, ok in checks.items():
    print(f'  {"✅" if ok else "❌"}  {label}')

if not checks['HF_TOKEN set']:
    print('\n  ⚠️  No HF_TOKEN — change --mode all → --mode gemma in the cell below')

In [ ]:
import glob
ASPECTS_CSV = glob.glob('aspects_*.csv')[0]

# No HF_TOKEN? Change --mode all  →  --mode gemma
!python response_generator.py \
    --aspects {ASPECTS_CSV} \
    --mode all

In [ ]:
import pandas as pd, glob, os

mfiles = glob.glob('metrics_*.csv')
if mfiles:
    print('=== FULL RUN METRICS ===')
    print(pd.read_csv(mfiles[0]).to_string(index=False))

print('\n=== OUTPUT FILES ===')
for f in sorted(os.listdir('.')):
    if f.startswith('responses_') or f.startswith('metrics_'):
        kb   = os.path.getsize(f) / 1024
        rows = len(pd.read_csv(f))
        print(f'  {f:<52} {kb:6.1f} KB  {rows} rows')

---
## F — Resume after session drop
Re-run A → B → C first to restore environment, then run this cell only.

In [ ]:
import os, glob, pandas as pd

ASPECTS_CSV = glob.glob('aspects_*.csv')[0]

ckpts = [f for f in os.listdir('.') if f.startswith('.ckpt_')]
if ckpts:
    print('Resuming from checkpoints:')
    for c in ckpts:
        mode = c.rsplit('_', 1)[-1].replace('.csv', '')
        done = len(pd.read_csv(c))
        print(f'  {mode:10s}: {done}/500 done  ({500-done} remaining)')
else:
    print('No checkpoints — starting fresh')

!python response_generator.py \
    --aspects {ASPECTS_CSV} \
    --mode all